# Le choix de la fenêtre d'observation

- **Objectif :** trouver l'intervalle optimal nécessaire pour prédire le décrochage scolaire.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from config import (
    CLEAN_ACTIVITES,
    CLEAN_EVALUATIONS,
    CLEAN_INSCRIPTIONS,
)

## 1. Préparation des données

### 1.1 Chargement des données

In [ ]:
activities_df = pd.read_csv(CLEAN_ACTIVITES)
evaluations_df = pd.read_csv(CLEAN_EVALUATIONS)
inscriptions_df = pd.read_csv(CLEAN_INSCRIPTIONS)

### 1.2 Conversion des colonnes temporelles

In [ ]:
inscriptions_df['date_debut'] = pd.to_datetime(inscriptions_df['date_debut'])
inscriptions_df['date_annulation'] = pd.to_datetime(inscriptions_df['date_annulation'])

evaluations_df['date_echeance'] = pd.to_datetime(evaluations_df['date_echeance'])
evaluations_df['date_soumission'] = pd.to_datetime(evaluations_df['date_soumission'])

activities_df['horodatage'] = pd.to_datetime(activities_df['horodatage'])

### 1.3 Jointure des tables d'événements

Pour exprimer chaque événement en jours écoulés depuis le début de l'inscription plutôt qu'en date absolue.

In [ ]:
# Ajout des colonnes date_debut et date_annulation aux DataFrames activities_df et evaluations_df
activities_merged = activities_df.merge(inscriptions_df[['id_inscription', 'date_debut', 'date_annulation','cohorte']], on='id_inscription', how='left')
evaluations_merged = evaluations_df.merge(inscriptions_df[['id_inscription', 'date_debut', 'date_annulation','cohorte']], on='id_inscription', how='left')

In [ ]:
# Ajout des colonnes days_since_start aux DataFrames activities_merged et evaluations_merged
activities_merged['days_since_start'] = (activities_merged['horodatage'] - activities_merged['date_debut']).dt.days
evaluations_merged['days_since_start'] = (evaluations_merged['date_soumission'] - evaluations_merged['date_debut']).dt.days

# Description des colonnes days_since_start
print("Activities days_since_start description:")
print(activities_merged['days_since_start'].describe())
print("\nEvaluations days_since_start description:")
print(evaluations_merged['days_since_start'].describe())

## 2. Visualisation

In [ ]:
sns.histplot(data=activities_merged, x='days_since_start', hue='cohorte', kde=True)
plt.xlabel('Days since start')
plt.ylabel('Frequency')
plt.show()

In [ ]:
sns.histplot(data=evaluations_merged, x='days_since_start', hue='cohorte', kde=True)
plt.xlabel('Days since start')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Verification des valeurs négatives dans les colonnes days_since_start
print((activities_merged['days_since_start'] < 0).sum())
print((evaluations_merged['days_since_start'] < 0).sum())
negative_eval = evaluations_merged[evaluations_merged['days_since_start'] < 0]
negative_eval['type_evaluation'].unique() # to drop

> Ils existent 22 observations avec des evaluations soumises avant le debut des programmes, ils doivent donc etre supprimees comme erreurs.

In [ ]:
activities_merged.groupby('cohorte')['days_since_start'].describe()

In [ ]:
evaluations_merged.groupby('cohorte')['days_since_start'].describe()

> Les donnees sont distribues de la meme maniere par rapport aux semestres.

## 3. Temps avant abandon

### 3.1 Distribution du temps de vie des inscriptions "Abandon"

In [ ]:
student_churn = inscriptions_df[inscriptions_df['resultat_final'] == 'Abandon']
student_churn['lifespan'] = (student_churn['date_annulation'] - student_churn['date_debut']).dt.days
student_churn.groupby('cohorte')['lifespan'].describe()

### 3.2 Visualisation

In [ ]:
sns.histplot(data=student_churn, x='lifespan', hue='cohorte', kde=True)
plt.xlabel('Lifespan (days)')
plt.ylabel('Frequency')
plt.title('Distribution of Lifespan for Churned Students by semester')
plt.show()

> Les abandons ont presque la meme distributions du temps avant abandonner sur les 2 semestres

## 4. Sélection des candidats T_obs

### 4.1 Vérification de la troncature liée à l'horizon des données

Un T_obs trop long pourrait dépasser la date limite d'extraction des données pour les inscriptions les plus tardives, ce qui fausserait les features de fenêtre pour ces cas sans lien avec leur comportement réel.

In [ ]:
data_cutoff = activities_df['horodatage'].max()
data_cutoff

In [ ]:
# Les étudiants dont la fenêtre d'observation est tronquée
students_df = inscriptions_df.copy()
for T_obs in [30, 60, 90]:
    students_df['window_end'] = students_df['date_debut'] + pd.Timedelta(days=T_obs)
    truncated = (students_df['window_end'] > data_cutoff).sum()
    print(f"T_obs: {T_obs} days, Truncated: {truncated} students")

### 4.2 Diagnostic de couverture et de risque de fuite par candidat

Pour chaque T_obs candidat (30, 60, 90 jours) : part des inscriptions sans activité/évaluation dans la fenêtre, et part des abandons déjà partis avant la fermeture de la fenêtre (indicateur direct de risque de fuite temporelle).

In [ ]:
# Diagnostique des candidats de la fenêtre d'observation
total_inscriptions = inscriptions_df['id_inscription'].nunique()
results = []
for T_obs in [30,60,90]:
    activities_window = activities_merged[activities_merged['days_since_start'] <= T_obs]
    activity_summary = activities_window.groupby('id_inscription').agg(
        total_activities=('id_activite', 'count'),
        total_volume=('volume', 'sum'),
    )
    evaluations_window = evaluations_merged[evaluations_merged['days_since_start'] <= T_obs]
    evaluation_summary = evaluations_window.groupby('id_inscription').agg(
        total_evaluations=('id_evaluation', 'count')
    )
    n_activity = activity_summary.shape[0]
    n_evaluation = evaluation_summary.shape[0]

    results.append({
        'T_obs': T_obs,
        'no activity %': 1 - n_activity / total_inscriptions,
        'no evaluation %': 1 - n_evaluation / total_inscriptions,
        'median activity': activity_summary['total_activities'].median(),
        'median evaluation': evaluation_summary['total_evaluations'].median(),
        'Churn %': (student_churn['lifespan'] <= T_obs).mean()
    })
diagnostics_df = pd.DataFrame(results)
display(diagnostics_df)

In [ ]:
# Cadence des évaluations
evaluations_merged['deadline_from_start'] = (evaluations_merged['date_echeance'] - evaluations_merged['date_debut']).dt.days
first_eval_deadline = evaluations_merged.groupby('id_inscription')['deadline_from_start'].min()
first_eval_deadline.describe()

> les dates d'echeance de quelques evaluations sont avant la date de debut des programmes.

In [ ]:
negative_deadline = evaluations_merged[evaluations_merged['date_echeance'] < evaluations_merged['date_debut']]
negative_deadline['id_evaluation'].count() #to drop

In [ ]:
soumission_anomaly_ids = evaluations_merged.loc[
    evaluations_merged['date_soumission'] < evaluations_merged['date_debut'], 'id_evaluation'
]
echeance_anomaly_ids = negative_deadline['id_evaluation']

overlap = set(soumission_anomaly_ids) & set(echeance_anomaly_ids)
print(len(overlap), len(soumission_anomaly_ids), len(echeance_anomaly_ids))

## Conclusion

- Les cohortes (Semestre 1 et Semestre 2) ont des distributions similaires de `days_since_start` et de temps avant abandon (médianes de 114 et 113 jours). Elles sont donc regroupées pour la suite de l'analyse.

- Le temps avant abandon a une médiane de ~113 jours, avec un premier quartile autour de 56-62 jours (calculé sur 3 972 des 4 012 abandons, les 40 restants n'ayant pas de `date_annulation`).

- Aucune inscription n'est tronquée par l'horizon des données jusqu'à T_obs = 90 jours.

- La première échéance d'évaluation programmée a une médiane de 78 jours après `date_debut`. La rareté des évaluations aux fenêtres courtes (97% sans évaluation à T_obs = 30) est donc surtout structurelle : le programme n'a simplement pas encore atteint ce jalon, plutôt qu'un signe de désengagement.

- Plus la fenêtre s'allonge, plus la couverture s'améliore (67% → 46% → 32% sans activité pour T_obs = 30/60/90), mais le risque de fuite augmente aussi : à T_obs = 90, 39.5% des abandons ont déjà quitté avant la fin de la fenêtre, contre 10.8% à T_obs = 30.

- T_obs = 60 jours est retenu comme fenêtre principale, comme compromis entre couverture et risque de fuite. T_obs = 90 jours reste une alternative à documenter si un meilleur rappel s'avère nécessaire, avec un risque de fuite plus élevé.

- 22 évaluations ont une `date_soumission` antérieure à `date_debut`, et 14 ont une `date_echeance` antérieure à `date_debut` (seulement 3 en commun entre les deux). Ces lignes seront exclues des calculs de features basées sur la fenêtre d'observation.